In [ ]:
#@title **📦 Install Dependencies & Setup Environment** { display-mode: "form" }

import subprocess
import sys

print("🔧 Installing system dependencies...")
try:
    # Install unrar for RAR file extraction
    subprocess.run(['apt-get', 'update', '-qq'], check=True, stdout=subprocess.DEVNULL)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'unrar'], check=True, stdout=subprocess.DEVNULL)
    print("✅ unrar installed successfully")
except subprocess.CalledProcessError as e:
    print(f"❌ Failed to install unrar: {e}")
    sys.exit(1)

print("\n🔧 Installing Python dependencies...")
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'requests', 'tqdm'], check=True)
    print("✅ Python packages installed successfully")
except subprocess.CalledProcessError as e:
    print(f"❌ Failed to install Python packages: {e}")
    sys.exit(1)

print("\n✅ Environment setup completed!")

In [ ]:
#@title **🛠️ Function Definitions** { display-mode: "form" }

import os
import shutil
import zipfile
import tarfile
import requests
import random
import json
import time
import re
from pathlib import Path
from tqdm import tqdm
from typing import List, Tuple, Optional

# Supported audio extensions
AUDIO_EXTENSIONS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a', '.aac', '.wma', '.aiff', '.ape'}

def extract_mozilla_dataset_id(url: str) -> Optional[str]:
    """
    Extract dataset ID from Mozilla Data Collective URL.
    
    Args:
        url: Mozilla URL (various formats accepted)
        
    Returns:
        Dataset ID or None if not found
    """
    # Pattern 1: https://datacollective.mozillafoundation.org/datasets/DATASET_ID
    # Pattern 2: https://datacollective.mozillafoundation.org/api/datasets/DATASET_ID/download
    # Pattern 3: https://datacollective.mozillafoundation.org/api/datasets/DATASET_ID
    
    patterns = [
        r'datasets/([a-zA-Z0-9]+)',  # Match any dataset ID after /datasets/
    ]
    
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    
    return None


def download_mozilla_dataset(
    mozilla_url: str,
    mozilla_api_key: str,
    output_path: Path
) -> bool:
    """
    Download dataset from Mozilla Data Collective using 2-step API process.
    
    Args:
        mozilla_url: Mozilla dataset URL (various formats accepted)
        mozilla_api_key: Mozilla API key
        output_path: Path to save the downloaded file
        
    Returns:
        bool: True if download successful, False otherwise
    """
    try:
        # Extract dataset ID from URL
        dataset_id = extract_mozilla_dataset_id(mozilla_url)
        
        if not dataset_id:
            print(f"❌ Could not extract dataset ID from URL: {mozilla_url}")
            print("💡 Please provide a valid Mozilla Data Collective URL")
            print("   Example: https://datacollective.mozillafoundation.org/datasets/DATASET_ID")
            return False
        
        print(f"🔍 Detected dataset ID: {dataset_id}")
        
        # Build API URL
        api_base_url = f"https://datacollective.mozillafoundation.org/api/datasets/{dataset_id}"
        
        # Step 1: POST request to get download token
        print(f"🔑 Requesting download token from Mozilla Data Collective...")
        post_url = f"{api_base_url}/download"
        
        headers = {
            "Authorization": f"Bearer {mozilla_api_key}",
            "Content-Type": "application/json"
        }
        
        try:
            response = requests.post(post_url, headers=headers, timeout=30)
            response.raise_for_status()
        except requests.exceptions.HTTPError as e:
            if response.status_code == 401:
                print(f"❌ Authentication failed. Please check your API key.")
                print("💡 Make sure your Mozilla API key is correct and active.")
            elif response.status_code == 404:
                print(f"❌ Dataset not found. Please check the dataset ID: {dataset_id}")
                print("💡 Verify the URL is correct and the dataset exists.")
            else:
                print(f"❌ HTTP Error {response.status_code}: {e}")
            return False
        except requests.exceptions.Timeout:
            print(f"❌ Request timed out. Please check your internet connection.")
            return False
        except requests.exceptions.ConnectionError:
            print(f"❌ Connection failed. Please check your internet connection.")
            return False
        
        # Extract download token from response
        try:
            response_data = response.json()
            download_token = response_data.get('token') or response_data.get('downloadToken')
        except json.JSONDecodeError:
            print(f"❌ Invalid response format from server.")
            print(f"💡 Response: {response.text[:200]}")
            return False
        
        if not download_token:
            print(f"❌ No download token received from server.")
            print(f"💡 Response: {response_data}")
            return False
        
        print(f"✅ Download token received")
        
        # Step 2: GET request with token to download actual file
        time.sleep(1)  # Brief pause between requests
        
        print(f"📥 Downloading dataset from Mozilla Data Collective...")
        get_url = f"{api_base_url}/download/{download_token}"
        
        try:
            response = requests.get(get_url, headers=headers, stream=True, timeout=120)
            response.raise_for_status()
        except requests.exceptions.HTTPError as e:
            print(f"❌ Download failed with HTTP Error {response.status_code}")
            print("💡 The download token may have expired. Please try again.")
            return False
        except requests.exceptions.Timeout:
            print(f"❌ Download timed out. The file may be too large.")
            print("💡 Please try again with a stable internet connection.")
            return False
        except requests.exceptions.ConnectionError:
            print(f"❌ Connection lost during download.")
            return False
        
        # Get filename from Content-Disposition header if available
        content_disposition = response.headers.get('Content-Disposition', '')
        if 'filename=' in content_disposition:
            filename = content_disposition.split('filename=')[-1].strip('"\'')
            output_path = output_path.parent / filename
            print(f"📦 Detected filename: {filename}")
        
        total_size = int(response.headers.get('content-length', 0))
        
        if total_size == 0:
            print("⚠️ Warning: Could not determine file size")
        
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        with open(output_path, 'wb') as f, tqdm(
            desc=output_path.name,
            total=total_size,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
        ) as pbar:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:  # filter out keep-alive chunks
                    size = f.write(chunk)
                    pbar.update(size)
        
        print(f"✅ Mozilla download completed: {output_path.name}")
        return True
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Network error occurred: {str(e)}")
        print("💡 Please check your internet connection and try again.")
        return False
    except Exception as e:
        print(f"❌ Unexpected error during Mozilla download: {str(e)}")
        print("💡 Please report this issue if it persists.")
        return False


def download_file(url: str, output_path: Path) -> bool:
    """
    Download file from URL with progress bar.
    
    Args:
        url: URL to download from
        output_path: Path to save the downloaded file
        
    Returns:
        bool: True if download successful, False otherwise
    """
    try:
        print(f"📥 Downloading from: {url}")
        response = requests.get(url, stream=True, timeout=30)
        response.raise_for_status()
        
        total_size = int(response.headers.get('content-length', 0))
        
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        with open(output_path, 'wb') as f, tqdm(
            desc=output_path.name,
            total=total_size,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
        ) as pbar:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:  # filter out keep-alive chunks
                    size = f.write(chunk)
                    pbar.update(size)
        
        print(f"✅ Download completed: {output_path}")
        return True
        
    except requests.exceptions.HTTPError as e:
        print(f"❌ HTTP Error {response.status_code}: {e}")
        print("💡 Please check if the URL is correct and accessible.")
        return False
    except requests.exceptions.Timeout:
        print(f"❌ Download timed out.")
        print("💡 Please check your internet connection and try again.")
        return False
    except requests.exceptions.ConnectionError:
        print(f"❌ Connection failed.")
        print("💡 Please check your internet connection.")
        return False
    except requests.exceptions.RequestException as e:
        print(f"❌ Download failed: {e}")
        return False
    except Exception as e:
        print(f"❌ Unexpected error during download: {e}")
        return False


def extract_archive(archive_path: Path, extract_to: Path) -> bool:
    """
    Extract ZIP, RAR, or TAR.GZ archive to specified directory.
    
    Args:
        archive_path: Path to the archive file
        extract_to: Directory to extract files to
        
    Returns:
        bool: True if extraction successful, False otherwise
    """
    try:
        extract_to.mkdir(parents=True, exist_ok=True)
        
        suffix = archive_path.suffix.lower()
        name_lower = archive_path.name.lower()
        
        if suffix == '.zip':
            print(f"📂 Extracting ZIP archive: {archive_path.name}")
            with zipfile.ZipFile(archive_path, 'r') as zip_ref:
                zip_ref.extractall(extract_to)
                
        elif suffix == '.rar':
            print(f"📂 Extracting RAR archive: {archive_path.name}")
            import subprocess
            result = subprocess.run(
                ['unrar', 'x', '-y', str(archive_path), str(extract_to)],
                capture_output=True,
                text=True
            )
            if result.returncode != 0:
                print(f"❌ RAR extraction failed: {result.stderr}")
                return False
        
        elif name_lower.endswith('.tar.gz') or name_lower.endswith('.tgz'):
            print(f"📂 Extracting TAR.GZ archive: {archive_path.name}")
            with tarfile.open(archive_path, 'r:gz') as tar_ref:
                tar_ref.extractall(extract_to)
        
        elif suffix == '.tar':
            print(f"📂 Extracting TAR archive: {archive_path.name}")
            with tarfile.open(archive_path, 'r') as tar_ref:
                tar_ref.extractall(extract_to)
                
        else:
            print(f"❌ Unsupported archive format: {suffix}")
            return False
        
        print(f"✅ Extraction completed to: {extract_to}")
        return True
        
    except zipfile.BadZipFile:
        print(f"❌ The ZIP file is corrupted or invalid.")
        print("💡 Please try downloading the file again.")
        return False
    except tarfile.ReadError:
        print(f"❌ The TAR/GZ file is corrupted or invalid.")
        print("💡 Please try downloading the file again.")
        return False
    except PermissionError:
        print(f"❌ Permission denied to extract files.")
        print("💡 Please check folder permissions.")
        return False
    except Exception as e:
        print(f"❌ Extraction failed: {str(e)}")
        print("💡 The archive file may be corrupted. Try downloading again.")
        return False


def scan_audio_files(directory: Path) -> List[Path]:
    """
    Recursively scan directory for audio files.
    
    Args:
        directory: Root directory to scan
        
    Returns:
        List of Path objects for audio files
    """
    audio_files = []
    
    print(f"🔍 Scanning for audio files in: {directory}")
    
    for file_path in directory.rglob('*'):
        if file_path.is_file() and file_path.suffix.lower() in AUDIO_EXTENSIONS:
            audio_files.append(file_path)
    
    print(f"✅ Found {len(audio_files)} audio files")
    return audio_files


def split_and_move_files(
    audio_files: List[Path],
    source_root: Path,
    target_root: Path,
    type_dataset: str,
    name_dataset: str,
    train_ratio: float = 0.8,
    val_ratio: float = 0.1,
    test_ratio: float = 0.1
) -> Tuple[int, int, int]:
    """
    Split audio files into train/validation/test sets and move to target structure.
    
    Args:
        audio_files: List of audio file paths
        source_root: Root directory of extracted files
        target_root: Root directory for dataset structure
        type_dataset: 'clean' or 'noise'
        name_dataset: Name of the dataset
        train_ratio: Ratio for training set (default 0.8)
        val_ratio: Ratio for validation set (default 0.1)
        test_ratio: Ratio for test set (default 0.1)
        
    Returns:
        Tuple of (train_count, val_count, test_count)
    """
    # Validate ratios
    total_ratio = train_ratio + val_ratio + test_ratio
    if abs(total_ratio - 1.0) > 0.001:
        print(f"⚠️ Warning: Ratios sum to {total_ratio}, normalizing...")
        train_ratio /= total_ratio
        val_ratio /= total_ratio
        test_ratio /= total_ratio
    
    # Shuffle files for random distribution
    random.shuffle(audio_files)
    
    total_files = len(audio_files)
    train_count = int(total_files * train_ratio)
    val_count = int(total_files * val_ratio)
    
    # Split indices
    train_files = audio_files[:train_count]
    val_files = audio_files[train_count:train_count + val_count]
    test_files = audio_files[train_count + val_count:]
    
    splits = {
        'train': train_files,
        'validation': val_files,
        'test': test_files
    }
    
    print(f"\n📊 Split summary:")
    print(f"   Train: {len(train_files)} files ({len(train_files)/total_files*100:.1f}%)")
    print(f"   Validation: {len(val_files)} files ({len(val_files)/total_files*100:.1f}%)")
    print(f"   Test: {len(test_files)} files ({len(test_files)/total_files*100:.1f}%)")
    
    # Move files to target structure
    for split_name, files in splits.items():
        print(f"\n📁 Processing {split_name} set...")
        
        for file_path in tqdm(files, desc=f"Moving {split_name} files"):
            # Get relative path from source root
            try:
                rel_path = file_path.relative_to(source_root)
            except ValueError:
                # If file is not relative to source_root, use just the filename
                rel_path = Path(file_path.name)
            
            # Build target path: dataset/{split}/{type}/{name}/{original_structure}
            target_path = target_root / split_name / type_dataset / name_dataset / rel_path
            
            # Create parent directories
            target_path.parent.mkdir(parents=True, exist_ok=True)
            
            # Move file
            shutil.copy2(file_path, target_path)
    
    print(f"\n✅ File splitting and moving completed!")
    return len(train_files), len(val_files), len(test_files)


def configure_git(username: str, email: str) -> bool:
    """
    Configure git global settings.
    
    Args:
        username: Git username
        email: Git email
        
    Returns:
        bool: True if configuration successful
    """
    try:
        import subprocess
        subprocess.run(['git', 'config', '--global', 'user.name', username], check=True)
        subprocess.run(['git', 'config', '--global', 'user.email', email], check=True)
        print(f"✅ Git configured: {username} <{email}>")
        return True
    except Exception as e:
        print(f"❌ Git configuration failed: {e}")
        return False


def push_to_github(
    repo_path: Path,
    commit_message: str,
    github_token: str = None,
    repo_url: str = None
) -> bool:
    """
    Commit and push changes to GitHub repository.
    
    Args:
        repo_path: Path to git repository
        commit_message: Commit message
        github_token: GitHub personal access token (optional)
        repo_url: Repository URL (optional, for setting remote)
        
    Returns:
        bool: True if push successful
    """
    try:
        import subprocess
        import sys
        
        os.chdir(repo_path)
        
        # Check if .git exists (repository already initialized)
        is_new_repo = not (repo_path / '.git').exists()
        
        # Initialize repo if needed
        if is_new_repo:
            subprocess.run(['git', 'init'], check=True)
            print("✅ Git repository initialized")
            
            if repo_url:
                # Configure remote with token if provided
                if github_token:
                    # Extract repo info from URL
                    if 'github.com/' in repo_url:
                        repo_part = repo_url.split('github.com/')[-1].replace('.git', '')
                        authenticated_url = f"https://{github_token}@github.com/{repo_part}.git"
                    else:
                        authenticated_url = repo_url
                    subprocess.run(['git', 'remote', 'add', 'origin', authenticated_url], check=True)
                else:
                    subprocess.run(['git', 'remote', 'add', 'origin', repo_url], check=True)
                print(f"✅ Remote origin set: {repo_url}")
        else:
            print("✅ Using existing git repository")
            
            # Update remote URL if token is provided
            if repo_url and github_token:
                if 'github.com/' in repo_url:
                    repo_part = repo_url.split('github.com/')[-1].replace('.git', '')
                    authenticated_url = f"https://{github_token}@github.com/{repo_part}.git"
                    
                    # Update remote URL
                    try:
                        subprocess.run(['git', 'remote', 'set-url', 'origin', authenticated_url], check=True)
                        print("✅ Remote URL updated with authentication")
                    except:
                        print("⚠️ Could not update remote URL")
        
        # Check repository size before adding
        print("\n📊 Calculating repository size...")
        total_size = 0
        file_count = 0
        large_files = []
        
        for root, dirs, files in os.walk(repo_path):
            # Skip .git directory
            if '.git' in root:
                continue
            for file in files:
                file_path = os.path.join(root, file)
                try:
                    file_size = os.path.getsize(file_path)
                    total_size += file_size
                    file_count += 1
                    
                    # Check for files larger than 100MB (GitHub limit)
                    if file_size > 100 * 1024 * 1024:
                        large_files.append((file_path, file_size))
                except:
                    pass
        
        print(f"📁 Files to commit: {file_count:,} files")
        print(f"💾 Total size: {total_size / (1024*1024):.2f} MB")
        
        # Warning for large files
        if large_files:
            print(f"\n⚠️ WARNING: Found {len(large_files)} file(s) larger than 100MB:")
            for file_path, size in large_files[:5]:  # Show first 5
                rel_path = os.path.relpath(file_path, repo_path)
                print(f"   • {rel_path}: {size / (1024*1024):.2f} MB")
            if len(large_files) > 5:
                print(f"   ... and {len(large_files) - 5} more files")
            print("\n💡 GitHub rejects files larger than 100MB!")
            print("💡 Solutions:")
            print("   1. Use Git LFS (Large File Storage): https://git-lfs.github.com/")
            print("   2. Split dataset into smaller chunks")
            print("   3. Use alternative hosting (Google Drive, Hugging Face, etc.)")
            print("\n❌ Cannot push files larger than 100MB without Git LFS")
            return False
        
        if total_size > 100 * 1024 * 1024:  # > 100 MB
            print("⚠️ Warning: Large repository detected. This may take a while...")
        
        if total_size > 1024 * 1024 * 1024:  # > 1 GB
            print("⚠️ WARNING: Repository is very large (>1GB)")
            print("💡 Consider using Git LFS or alternative hosting for large datasets")
        
        # Determine current branch
        try:
            branch_result = subprocess.run(
                ['git', 'rev-parse', '--abbrev-ref', 'HEAD'],
                capture_output=True,
                text=True,
                check=True
            )
            current_branch = branch_result.stdout.strip()
            print(f"\n📍 Current branch: {current_branch}")
        except:
            current_branch = 'main'
            print(f"\n📍 Using default branch: {current_branch}")
        
        # Check if remote branch exists
        remote_branch_exists = False
        if not is_new_repo:
            print("\n📥 Checking remote repository status...")
            print("=" * 60)
            
            # Fetch remote
            try:
                subprocess.run(['git', 'fetch', 'origin'], check=True, capture_output=True)
                print("✅ Fetched remote changes")
                
                # Check if remote branch exists
                remote_check = subprocess.run(
                    ['git', 'ls-remote', '--heads', 'origin', current_branch],
                    capture_output=True,
                    text=True
                )
                
                if remote_check.stdout.strip():
                    remote_branch_exists = True
                    print(f"✅ Remote branch 'origin/{current_branch}' exists")
                else:
                    print(f"ℹ️ Remote branch 'origin/{current_branch}' doesn't exist yet")
                    print("   This is a first push to this branch")
                
            except subprocess.CalledProcessError:
                print("⚠️ Could not fetch remote (repository might be empty or new)")
            
            # PULL from remote if branch exists
            if remote_branch_exists:
                try:
                    pull_result = subprocess.run(
                        ['git', 'pull', '--rebase', 'origin', current_branch],
                        capture_output=True,
                        text=True,
                        timeout=60
                    )
                    
                    if pull_result.returncode == 0:
                        print(f"✅ Pulled and rebased with origin/{current_branch}")
                    else:
                        print("⚠️ Rebase failed, trying regular pull...")
                        pull_result = subprocess.run(
                            ['git', 'pull', 'origin', current_branch],
                            capture_output=True,
                            text=True,
                            timeout=60
                        )
                        if pull_result.returncode == 0:
                            print(f"✅ Pulled from origin/{current_branch}")
                        else:
                            print(f"⚠️ Pull failed: {pull_result.stderr[:100]}")
                except subprocess.TimeoutExpired:
                    print("⚠️ Pull timed out, continuing anyway...")
                except Exception as e:
                    print(f"⚠️ Pull failed: {str(e)[:100]}, continuing anyway...")
            
            print("=" * 60)
        
        # Add all changes with progress
        print("\n➕ Adding files to git...")
        subprocess.run(['git', 'add', '.'], check=True)
        print("✅ Files staged for commit")
        
        # Commit
        print(f"\n💾 Committing changes...")
        result = subprocess.run(
            ['git', 'commit', '-m', commit_message],
            capture_output=True,
            text=True
        )
        
        has_new_commit = False
        has_unpushed_commits = False
        
        if result.returncode == 0:
            print(f"✅ Changes committed: {commit_message}")
            has_new_commit = True
            has_unpushed_commits = True
        else:
            if "nothing to commit" in result.stdout.lower() or "nothing to commit" in result.stderr.lower():
                print("ℹ️ No new changes to commit")
                
                # Check for existing commits in the repository
                try:
                    log_result = subprocess.run(
                        ['git', 'log', '--oneline', '-1'],
                        capture_output=True,
                        text=True
                    )
                    
                    if log_result.returncode == 0 and log_result.stdout.strip():
                        print(f"📋 Latest commit: {log_result.stdout.strip()}")
                        
                        # Check if there are unpushed commits (only if remote branch exists)
                        if remote_branch_exists:
                            try:
                                unpushed_result = subprocess.run(
                                    ['git', 'log', f'origin/{current_branch}..HEAD', '--oneline'],
                                    capture_output=True,
                                    text=True
                                )
                                
                                if unpushed_result.stdout.strip():
                                    print(f"📋 Found unpushed commits:")
                                    for line in unpushed_result.stdout.strip().split('\n')[:5]:
                                        print(f"   {line}")
                                    has_unpushed_commits = True
                                else:
                                    print("✅ All commits are already pushed to remote")
                                    return True
                            except subprocess.CalledProcessError:
                                print("⚠️ Could not check for unpushed commits")
                                has_unpushed_commits = True
                        else:
                            # Remote branch doesn't exist, so we need to push
                            print("🔄 Local commits exist but remote branch doesn't exist yet")
                            has_unpushed_commits = True
                    else:
                        print("⚠️ No commits found in repository")
                        print("💡 The repository appears to be empty. Nothing to push.")
                        return True
                        
                except subprocess.CalledProcessError:
                    print("⚠️ Could not check commit history")
                    return False
            else:
                print(f"⚠️ Commit error: {result.stdout or result.stderr}")
                return False
        
        # Only push if we have commits to push
        if not has_unpushed_commits:
            print("ℹ️ Nothing to push")
            return True
        
        # Push to remote with real-time output
        print("\n📤 Pushing to GitHub (this may take a while for large datasets)...")
        print("=" * 60)
        
        # Try pushing with progress
        push_process = subprocess.Popen(
            ['git', 'push', '-u', 'origin', current_branch, '--progress'],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            universal_newlines=True,
            bufsize=1
        )
        
        # Print output in real-time
        push_output = []
        for line in iter(push_process.stdout.readline, ''):
            if line:
                line = line.strip()
                push_output.append(line)
                if line:
                    print(f"  {line}")
                    sys.stdout.flush()
        
        push_process.wait()
        
        # If current branch push failed, try main/master
        if push_process.returncode != 0:
            if current_branch not in ['main', 'master']:
                print(f"\n🔄 Branch '{current_branch}' failed, trying 'main'...")
                print("=" * 60)
                
                push_process = subprocess.Popen(
                    ['git', 'push', '-u', 'origin', 'main', '--progress'],
                    stdout=subprocess.PIPE,
                    stderr=subprocess.STDOUT,
                    universal_newlines=True,
                    bufsize=1
                )
                
                push_output = []
                for line in iter(push_process.stdout.readline, ''):
                    if line:
                        line = line.strip()
                        push_output.append(line)
                        if line:
                            print(f"  {line}")
                            sys.stdout.flush()
                
                push_process.wait()
            
            # If main failed, try master
            if push_process.returncode != 0 and current_branch != 'master':
                print("\n🔄 Trying 'master' branch...")
                print("=" * 60)
                
                push_process = subprocess.Popen(
                    ['git', 'push', '-u', 'origin', 'master', '--progress'],
                    stdout=subprocess.PIPE,
                    stderr=subprocess.STDOUT,
                    universal_newlines=True,
                    bufsize=1
                )
                
                push_output = []
                for line in iter(push_process.stdout.readline, ''):
                    if line:
                        line = line.strip()
                        push_output.append(line)
                        if line:
                            print(f"  {line}")
                            sys.stdout.flush()
                
                push_process.wait()
        
        print("=" * 60)
        
        if push_process.returncode == 0:
            print("\n✅ Successfully pushed to GitHub!")
            return True
        else:
            print(f"\n❌ Push failed")
            if push_output:
                print("\n📋 Error details:")
                for line in push_output[-10:]:  # Show last 10 lines
                    print(f"  {line}")
            
            # Check for common errors
            error_text = '\n'.join(push_output).lower()
            if 'large file' in error_text or 'file size' in error_text:
                print("\n💡 The error is related to large files.")
                print("   Please use Git LFS or split your dataset into smaller chunks.")
            elif 'authentication' in error_text or 'permission' in error_text or '403' in error_text:
                print("\n💡 Authentication failed.")
                print("   Please check your GitHub token and repository permissions.")
            elif 'connection' in error_text or 'network' in error_text:
                print("\n💡 Network connection issue.")
                print("   Please check your internet connection and try again.")
            elif 'rejected' in error_text or 'non-fast-forward' in error_text:
                print("\n💡 Push was rejected (possibly due to remote changes).")
                print("   The pull step should have handled this, but there may be conflicts.")
                print("   Try running the push stage again.")
            
            return False
            
    except Exception as e:
        print(f"❌ GitHub push failed: {e}")
        import traceback
        traceback.print_exc()
        return False

print("✅ All functions loaded successfully!")

In [ ]:
#@title **🚀 Execute Pipeline** { display-mode: "form" }

#@markdown ---
#@markdown ### 📝 Dataset Name
name_dataset = "" #@param {type:"string"}

#@markdown ---
#@markdown ### 📥 Download Source (Choose ONE method below)

#@markdown #### Method 1: Direct URL Download
url_dataset = "" #@param {type:"string"}

#@markdown #### Method 2: Mozilla Data Collective
#@markdown Paste any Mozilla URL format (e.g., https://datacollective.mozillafoundation.org/datasets/DATASET_ID)
mozilla_url = "" #@param {type:"string"}
mozilla_api_key = "" #@param {type:"string"}

#@markdown ---
#@markdown ### 🏷️ Dataset Type
type_dataset = "clean" #@param ["clean", "noise"]

#@markdown ---
#@markdown ### ⚙️ Pipeline Stages
run_download = True #@param {type:"boolean"}
run_splitting = True #@param {type:"boolean"}
run_push_github = False #@param {type:"boolean"}

#@markdown ---
#@markdown ### 🔐 GitHub Configuration (Use Separate Repos for Clean/Noise)
github_username = "" #@param {type:"string"}
github_email = "" #@param {type:"string"}
github_token = "" #@param {type:"string"}

#@markdown #### Repository URLs (one for each type)
github_repo_clean = "" #@param {type:"string"}
github_repo_noise = "" #@param {type:"string"}

commit_message = "Add dataset" #@param {type:"string"}

#@markdown ---

# ============================================================================
# MAIN EXECUTION
# ============================================================================

print("=" * 60)
print("🎯 DATASET PIPELINE EXECUTION")
print("=" * 60)

# Validate dataset name only if needed for download or splitting
if (run_download or run_splitting) and not name_dataset:
    print("\n❌ Error: Dataset name is required for download/splitting stages!")
    print("💡 Please fill in the 'name_dataset' field above.")
    print("💡 Or disable 'run_download' and 'run_splitting' if you only want to push.")
    import sys
    sys.exit(1)

# Validate download source only if download stage is enabled
if run_download:
    use_mozilla = bool(mozilla_url and mozilla_api_key)
    use_direct = bool(url_dataset)

    if not use_mozilla and not use_direct:
        print("\n❌ Error: No download source specified!")
        print("💡 Please provide either:")
        print("   • Direct URL in 'url_dataset' field, OR")
        print("   • Mozilla URL + API Key in 'mozilla_url' and 'mozilla_api_key' fields")
        import sys
        sys.exit(1)

    if use_mozilla and use_direct:
        print("\n⚠️ Warning: Both download methods specified.")
        print("🔵 Prioritizing Mozilla Data Collective download")
        use_direct = False

# Display configuration
print(f"\n📋 Configuration:")
if name_dataset:
    print(f"   Dataset Name: {name_dataset}")
    print(f"   Dataset Type: {type_dataset}")
if run_download:
    print(f"   Download Method: {'Mozilla Data Collective' if use_mozilla else 'Direct URL'}")
print(f"   Stages: Download={run_download}, Split={run_splitting}, GitHub={run_push_github}")

# Setup paths - dataset will be in /content/dataset, NOT /content/dataset/clean
from pathlib import Path

base_dir = Path('/content')
temp_dir = base_dir / 'temp'
dataset_dir = base_dir / 'dataset'  # Root dataset directory
download_path = temp_dir / 'downloaded_archive'

print(f"\n📁 Working directories:")
print(f"   Base: {base_dir}")
print(f"   Temp: {temp_dir}")
print(f"   Dataset: {dataset_dir}")

# ============================================================================
# STAGE 1: DOWNLOAD
# ============================================================================

if run_download:
    print("\n" + "=" * 60)
    print("📥 STAGE 1: DOWNLOADING DATASET")
    print("=" * 60)
    
    # Clean temp directory if exists
    if temp_dir.exists():
        print(f"\n🧹 Cleaning previous temp directory...")
        shutil.rmtree(temp_dir)
    temp_dir.mkdir(parents=True)
    
    if use_mozilla:
        # Mozilla Data Collective download
        print("🔵 Using Mozilla Data Collective API\n")
        download_path = temp_dir / 'archive.tar.gz'
        success = download_mozilla_dataset(mozilla_url, mozilla_api_key, download_path)
        
        # Check if file was downloaded with different name
        if success:
            # Look for any downloaded file in temp directory
            downloaded_files = list(temp_dir.glob('*'))
            if downloaded_files:
                download_path = downloaded_files[0]
                print(f"📦 Using downloaded file: {download_path.name}")
    else:
        # Direct URL download
        print("🌐 Using direct URL download\n")
        from urllib.parse import urlparse
        parsed_url = urlparse(url_dataset)
        url_path = parsed_url.path
        
        if url_path.lower().endswith('.zip'):
            download_path = temp_dir / 'archive.zip'
        elif url_path.lower().endswith('.rar'):
            download_path = temp_dir / 'archive.rar'
        elif url_path.lower().endswith('.tar.gz') or url_path.lower().endswith('.tgz'):
            download_path = temp_dir / 'archive.tar.gz'
        else:
            # Default to zip
            download_path = temp_dir / 'archive.zip'
            print(f"⚠️ Warning: Could not detect archive type from URL, assuming ZIP\n")
        
        success = download_file(url_dataset, download_path)
    
    if not success:
        print("\n" + "=" * 60)
        print("❌ PIPELINE FAILED: Download Error")
        print("=" * 60)
        print("\n💡 Troubleshooting tips:")
        print("   1. Check your internet connection")
        print("   2. Verify the URL/API key is correct")
        print("   3. Ensure the file is accessible")
        print("   4. Try running the cell again")
        import sys
        sys.exit(1)
    
    # Verify file exists and has content
    if not download_path.exists():
        print(f"\n❌ Downloaded file not found: {download_path}")
        print("💡 The download may have failed silently. Please try again.")
        import sys
        sys.exit(1)
    
    file_size = download_path.stat().st_size
    if file_size == 0:
        print(f"\n❌ Downloaded file is empty (0 bytes)")
        print("💡 The download was incomplete. Please try again.")
        import sys
        sys.exit(1)
    
    print(f"\n✅ File verified: {download_path.name} ({file_size / (1024*1024):.2f} MB)")
    
    # Extract archive
    print(f"\n📂 Extracting archive...")
    extract_dir = temp_dir / 'extracted'
    success = extract_archive(download_path, extract_dir)
    
    if not success:
        print("\n" + "=" * 60)
        print("❌ PIPELINE FAILED: Extraction Error")
        print("=" * 60)
        print("\n💡 The archive file may be corrupted.")
        print("   Try downloading the file again.")
        import sys
        sys.exit(1)
    
    # Find the actual root directory (handle nested folders)
    extracted_contents = list(extract_dir.iterdir())
    
    if not extracted_contents:
        print(f"\n❌ No files found after extraction")
        print("💡 The archive may be empty or corrupted.")
        import sys
        sys.exit(1)
    
    if len(extracted_contents) == 1 and extracted_contents[0].is_dir():
        source_root = extracted_contents[0]
        print(f"📁 Using nested directory as source root: {source_root.name}")
    else:
        source_root = extract_dir
        print(f"📁 Using extraction directory as source root")
    
    print(f"\n✅ STAGE 1 COMPLETED: Download & Extraction")
else:
    print("\n⏭️ STAGE 1: SKIPPED (run_download = False)")

# ============================================================================
# STAGE 2: SPLITTING
# ============================================================================

if run_splitting:
    print("\n" + "=" * 60)
    print("✂️ STAGE 2: SPLITTING DATASET")
    print("=" * 60)
    
    # Check if we need to find the source root
    if not run_download:
        # User skipped download, look for existing extracted files
        extract_dir = temp_dir / 'extracted'
        
        if not extract_dir.exists():
            print("\n❌ Error: No extracted files found!")
            print("💡 Please enable 'run_download' first, or manually place extracted files in:")
            print(f"   {extract_dir}")
            import sys
            sys.exit(1)
        
        extracted_contents = list(extract_dir.iterdir())
        
        if not extracted_contents:
            print(f"\n❌ No files found in extraction directory")
            print(f"💡 Please ensure files are in: {extract_dir}")
            import sys
            sys.exit(1)
        
        if len(extracted_contents) == 1 and extracted_contents[0].is_dir():
            source_root = extracted_contents[0]
            print(f"📁 Found source root: {source_root.name}")
        else:
            source_root = extract_dir
            print(f"📁 Using extraction directory as source root")
    
    # Scan for audio files
    audio_files = scan_audio_files(source_root)
    
    if not audio_files:
        print("\n" + "=" * 60)
        print("❌ PIPELINE FAILED: No Audio Files Found")
        print("=" * 60)
        print("\n💡 Possible reasons:")
        print("   1. The archive doesn't contain audio files")
        print("   2. Audio files are in an unsupported format")
        print(f"   3. Supported formats: {', '.join(AUDIO_EXTENSIONS)}")
        print(f"\n📁 Searched in: {source_root}")
        print("💡 You can check the extracted files manually in Colab's file browser.")
        import sys
        sys.exit(1)
    
    # Split and move files with type subfolder
    try:
        # Structure: dataset/{split}/{type}/{name}/
        train_count, val_count, test_count = split_and_move_files(
            audio_files=audio_files,
            source_root=source_root,
            target_root=dataset_dir,
            type_dataset=type_dataset,
            name_dataset=name_dataset
        )
        
        print(f"\n📊 Final dataset structure for '{type_dataset}':")
        print(f"   {dataset_dir}/train/{type_dataset}/{name_dataset}/: {train_count} files")
        print(f"   {dataset_dir}/validation/{type_dataset}/{name_dataset}/: {val_count} files")
        print(f"   {dataset_dir}/test/{type_dataset}/{name_dataset}/: {test_count} files")
        
        print(f"\n✅ STAGE 2 COMPLETED: Dataset Split")
    except Exception as e:
        print(f"\n❌ Error during file splitting: {str(e)}")
        print("💡 There may be issues with file permissions or disk space.")
        import sys
        sys.exit(1)
else:
    print("\n⏭️ STAGE 2: SKIPPED (run_splitting = False)")

# ============================================================================
# STAGE 3: GITHUB PUSH
# ============================================================================

if run_push_github:
    print("\n" + "=" * 60)
    print("📤 STAGE 3: PUSHING TO GITHUB")
    print("=" * 60)
    
    # Determine which repository to use based on type_dataset
    if type_dataset == 'clean':
        github_repo_url = github_repo_clean
        repo_type_name = "Clean Dataset Repository"
    elif type_dataset == 'noise':
        github_repo_url = github_repo_noise
        repo_type_name = "Noise Dataset Repository"
    else:
        print(f"\n❌ Error: Unknown dataset type '{type_dataset}'")
        import sys
        sys.exit(1)
    
    if not github_repo_url:
        print(f"\n❌ GitHub repository URL is required for {type_dataset} dataset!")
        print(f"💡 Please fill in the 'github_repo_{type_dataset}' field above.")
        import sys
        sys.exit(1)
    
    print(f"\n📋 Target Repository: {repo_type_name}")
    print(f"   URL: {github_repo_url}")
    print(f"   Type: {type_dataset}")
    
    # Validate dataset directory exists
    if not dataset_dir.exists():
        print("\n❌ Error: Dataset directory not found!")
        print(f"💡 Expected location: {dataset_dir}")
        print("💡 Please run 'run_splitting' stage first to create the dataset structure.")
        import sys
        sys.exit(1)
    
    # Check if the specific type subdirectory exists in each split
    # Expected: dataset/train/clean/, dataset/validation/clean/, dataset/test/clean/
    type_exists = False
    for split in ['train', 'validation', 'test']:
        split_type_dir = dataset_dir / split / type_dataset
        if split_type_dir.exists():
            type_exists = True
            break
    
    if not type_exists:
        print(f"\n❌ Error: No {type_dataset} dataset found!")
        print(f"💡 Expected structure: {dataset_dir}/train/{type_dataset}/{name_dataset}/")
        print(f"💡 Please run 'run_splitting' stage with type_dataset='{type_dataset}' first.")
        import sys
        sys.exit(1)
    
    # Restructure for GitHub: Remove type subfolder (clean/noise)
    # From: dataset/train/clean/name/ -> To: github_temp/clean/train/name/
    print(f"\n🔄 Restructuring {type_dataset} dataset for GitHub...")
    github_temp_dir = base_dir / 'github_temp' / type_dataset
    
    # Clean github temp if exists
    if github_temp_dir.exists():
        shutil.rmtree(github_temp_dir)
    github_temp_dir.mkdir(parents=True)
    
    # Copy files with new structure (removing type subfolder)
    restructured_count = 0
    for split in ['train', 'validation', 'test']:
        source_split = dataset_dir / split / type_dataset
        target_split = github_temp_dir / split
        
        if source_split.exists():
            print(f"   📁 Copying {split}...")
            # Copy everything under dataset/train/clean/* to github_temp/clean/train/*
            shutil.copytree(source_split, target_split, dirs_exist_ok=True)
            
            # Count files
            file_count_split = sum(1 for _ in target_split.rglob('*') if _.is_file())
            print(f"   ✅ {split}: {file_count_split} files")
            restructured_count += file_count_split
        else:
            print(f"   ⚠️ {split}: not found (skipped)")
    
    if restructured_count == 0:
        print("\n❌ Error: No files found after restructuring!")
        print(f"💡 Please check that {type_dataset} dataset exists in: {dataset_dir}")
        import sys
        sys.exit(1)
    
    print(f"\n📊 Dataset status:")
    print(f"   Total files to push: {restructured_count}")
    print(f"   Local structure: dataset/train/{type_dataset}/{name_dataset}/")
    print(f"   GitHub structure: train/{name_dataset}/ (without '{type_dataset}' folder)")
    
    if github_username and github_email:
        configure_git(github_username, github_email)
    else:
        print("⚠️ Warning: GitHub username/email not provided, using default git config")
    
    # Determine commit message
    if not commit_message:
        commit_message = f"Add {name_dataset} ({type_dataset}) dataset"
    else:
        commit_message = f"{commit_message} ({type_dataset})"
    
    # Push restructured dataset to GitHub
    print(f"\n🔄 Pushing to {type_dataset} repository...")
    success = push_to_github(
        repo_path=github_temp_dir,
        commit_message=commit_message,
        github_token=github_token if github_token else None,
        repo_url=github_repo_url
    )
    
    if success:
        print(f"\n✅ STAGE 3 COMPLETED: Pushed to GitHub")
        print(f"💡 {type_dataset.capitalize()} dataset pushed to separate repository")
        print(f"   Repository: {github_repo_url}")
        print(f"   Structure (in GitHub):")
        print(f"   • train/{name_dataset}/")
        print(f"   • validation/{name_dataset}/")
        print(f"   • test/{name_dataset}/")
    else:
        print("\n⚠️ GitHub push encountered issues")
        print("💡 Common reasons:")
        print("   • Files too large (GitHub limit: 100MB per file)")
        print("   • Repository too large (>1GB needs Git LFS)")
        print("   • Authentication issues (check token/credentials)")
        print("   • Network connectivity problems")
        print(f"\n💡 You can manually inspect the dataset directory: {github_temp_dir}")
else:
    print("\n⏭️ STAGE 3: SKIPPED (run_push_github = False)")

# ============================================================================
# CLEANUP
# ============================================================================

if run_download:
    print("\n" + "=" * 60)
    print("🧹 CLEANUP")
    print("=" * 60)

    try:
        if temp_dir.exists():
            shutil.rmtree(temp_dir)
            print("✅ Temporary files cleaned up")
    except Exception as e:
        print(f"⚠️ Warning: Could not clean temp directory: {e}")
        print("💡 You can manually delete the 'temp' folder if needed.")

# Clean github temp directory if push was successful
if run_push_github:
    try:
        github_temp_base = base_dir / 'github_temp'
        if github_temp_base.exists():
            shutil.rmtree(github_temp_base)
            print("✅ GitHub temp files cleaned up")
    except Exception as e:
        print(f"⚠️ Warning: Could not clean github temp directory: {e}")

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "=" * 60)
print("🎉 PIPELINE COMPLETED SUCCESSFULLY!")
print("=" * 60)

executed_stages = []
if run_download:
    executed_stages.append("Download & Extract")
if run_splitting:
    executed_stages.append("Split Dataset")
if run_push_github:
    executed_stages.append(f"Push to GitHub ({type_dataset})")

print(f"\n✅ Executed stages: {', '.join(executed_stages) if executed_stages else 'None'}")
print(f"📁 Dataset location: {dataset_dir}")
if name_dataset:
    print(f"📊 Dataset: {name_dataset} ({type_dataset})")

if run_push_github:
    if type_dataset == 'clean':
        print(f"🔗 Clean Repository: {github_repo_clean}")
    elif type_dataset == 'noise':
        print(f"🔗 Noise Repository: {github_repo_noise}")

print(f"\n💡 Next steps:")
if run_download and not run_splitting:
    print(f"   • Enable 'run_splitting' to organize the dataset")
elif run_splitting and not run_push_github:
    print(f"   • Enable 'run_push_github' to push to GitHub")
    print(f"   • Check the dataset structure in '{dataset_dir}'")
elif not any([run_download, run_splitting, run_push_github]):
    print(f"   • Enable at least one stage to process the dataset")
else:
    print(f"   • Your {type_dataset} dataset is ready!")
    if type_dataset == 'clean' and github_repo_noise:
        print(f"   • To upload noise dataset, change 'type_dataset' to 'noise' and run again")
    elif type_dataset == 'noise' and github_repo_clean:
        print(f"   • To upload clean dataset, change 'type_dataset' to 'clean' and run again")